In [ ]:
import sys
sys.path.insert(0, "..")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.nn import functional as F

from app.vjepa_droid.transforms import make_transforms
from utils.mpc_utils import (
    compute_new_pose,
    poses_to_diff
)

In [ ]:
# Settings. Each one can be overridden with an environment variable (VJEPA_DEVICE, VJEPA_DTYPE, ...).
# By default the notebook runs on the GPU in bfloat16 when there is one, otherwise as before on the CPU.
import os

DEVICE = os.environ.get("VJEPA_DEVICE") or ("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = getattr(torch, os.environ.get("VJEPA_DTYPE") or ("bfloat16" if DEVICE == "cuda" else "float32"))
# Energy grid: GRID_POINTS moves per axis from -GRID_SIZE to +GRID_SIZE metres, BATCH_SIZE moves at a time
GRID_SIZE = float(os.environ.get("VJEPA_GRID_SIZE") or 0.075)
GRID_POINTS = int(os.environ.get("VJEPA_GRID_POINTS") or 5)
BATCH_SIZE = int(os.environ.get("VJEPA_BATCH_SIZE") or 125)
CEM_SAMPLES = int(os.environ.get("VJEPA_CEM_SAMPLES") or 25)
CEM_STEPS = int(os.environ.get("VJEPA_CEM_STEPS") or 2)
SEED = int(os.environ.get("VJEPA_SEED") or 0)
RESULTS_DIR = os.environ.get("VJEPA_RESULTS_DIR")  # when set, the measurements are also saved there as JSON

torch.manual_seed(SEED)
np.random.seed(SEED)


def autocast():
    """Let bfloat16 weights take the float32 states and actions; does nothing in float32."""
    return torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=DTYPE == torch.bfloat16)

import json
import resource
import time
from contextlib import contextmanager

measurements = {"device": DEVICE, "dtype": str(DTYPE).removeprefix("torch.")}


@contextmanager
def measure(name):
    """Record the wall time of a block and, on the GPU, its peak memory."""
    if DEVICE == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    yield
    if DEVICE == "cuda":
        torch.cuda.synchronize()
        measurements[f"{name}_peak_vram_gib"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
    measurements[f"{name}_s"] = round(time.perf_counter() - start, 2)


print(f"device={DEVICE} dtype={DTYPE} grid={GRID_POINTS}^3 moves within +-{GRID_SIZE} m "
      f"cem_samples={CEM_SAMPLES} cem_steps={CEM_STEPS} seed={SEED}")

In [ ]:
# Initialize VJEPA 2-AC model from this checkout; the weights are downloaded to TORCH_HOME on first use
with measure("load"):
    encoder, predictor = torch.hub.load("..", "vjepa2_ac_vit_giant", source="local")
    encoder = encoder.to(DEVICE, DTYPE).eval()
    predictor = predictor.to(DEVICE, DTYPE).eval()

# Initialize transform
crop_size = 256
tokens_per_frame = int((crop_size // encoder.patch_size) ** 2)
transform = make_transforms(
    random_horizontal_flip=False,
    random_resize_aspect_ratio=(1., 1.),
    random_resize_scale=(1., 1.),
    reprob=0.,
    auto_augment=False,
    motion_shift=False,
    crop_size=crop_size,
)

In [ ]:
# Load robot trajectory

play_in_reverse = False  # Use this FLAG to try loading the trajectory backwards, and see how the energy landscape changes

trajectory = np.load("franka_example_traj.npz")
np_clips = trajectory["observations"]
np_states = trajectory["states"]
if play_in_reverse:
    np_clips = trajectory["observations"][:, ::-1].copy()
    np_states = trajectory["states"][:, ::-1].copy()
np_actions = np.expand_dims(poses_to_diff(np_states[0, 0], np_states[0, 1]), axis=(0, 1))

# Convert trajectory to torch tensors
clips = transform(np_clips[0]).unsqueeze(0).to(DEVICE)
states = torch.tensor(np_states).to(DEVICE)
actions = torch.tensor(np_actions)
print(f"clips: {clips.shape}; states: {states.shape}; actions: {actions.shape}")

In [ ]:
# Visualize loaded video frames from traj

T = len(np_clips[0])
plt.figure(figsize=(20, 3))
_ = plt.imshow(np.transpose(np_clips[0], (1, 0, 2, 3)).reshape(256, 256 * T, 3))

In [ ]:
def forward_target(c, normalize_reps=True):
    B, C, T, H, W = c.size()
    c = c.permute(0, 2, 1, 3, 4).flatten(0, 1).unsqueeze(2).repeat(1, 1, 2, 1, 1)
    h = encoder(c)
    h = h.view(B, T, -1, h.size(-1)).flatten(1, 2)
    if normalize_reps:
        h = F.layer_norm(h, (h.size(-1),))
    return h


def make_action_grid(grid_size, nsamples):
    """End-effector moves on an nsamples^3 grid from -grid_size to +grid_size metres in x, y and z, without
    rotation and with the gripper unchanged; x varies slowest and z fastest. Returns [nsamples^3, 1, 7]."""
    axis = torch.linspace(-grid_size, grid_size, nsamples)
    dx, dy, dz = torch.meshgrid(axis, axis, axis, indexing="ij")
    moves = torch.stack([dx, dy, dz], dim=-1).reshape(-1, 3)
    return torch.cat([moves, torch.zeros(len(moves), 4)], dim=-1).unsqueeze(1)


def forward_actions(z, action_samples, normalize_reps=True, action_repeat=1):
    """From the first frame of z, predict the representation after each action in action_samples [S, 1, 7]."""

    def step_predictor(_z, _a, _s):
        _z = predictor(_z, _a, _s)[:, -tokens_per_frame:]
        if normalize_reps:
            _z = F.layer_norm(_z, (_z.size(-1),))
        _s = compute_new_pose(_s[:, -1:], _a[:, -1:])
        return _z, _s

    # Context frame rep and context pose
    z_hat = z[:, :tokens_per_frame].repeat(len(action_samples), 1, 1)  # [S, N, D]
    s_hat = states[:, :1].repeat((len(action_samples), 1, 1))  # [S, 1, 7]
    a_hat = action_samples  # [S, 1, 7]

    for _ in range(action_repeat):
        _z, _s = step_predictor(z_hat, a_hat, s_hat)
        z_hat = torch.cat([z_hat, _z], dim=1)
        s_hat = torch.cat([s_hat, _s], dim=1)
        a_hat = torch.cat([a_hat, action_samples], dim=1)

    return z_hat, s_hat, a_hat

def loss_fn(z, h):
    z, h = z[:, -tokens_per_frame:], h[:, -tokens_per_frame:]
    loss = torch.abs(z - h)  # [B, N, D]
    loss = torch.mean(loss, dim=[1, 2])
    return loss.tolist()


def energy(h, action_samples):
    """Energy of each action: the L1 distance between the predicted next frame and the goal frame (the last
    frame of h). Computed BATCH_SIZE actions at a time, so that large grids fit in memory."""
    energies = []
    for chunk in action_samples.split(BATCH_SIZE):
        z_hat, _, _ = forward_actions(h, chunk.to(h.device, h.dtype))
        energies += loss_fn(z_hat, h)
    return energies

In [ ]:
# Compute energy for a cartesian grid of GRID_POINTS x GRID_POINTS x GRID_POINTS end-effector moves,
# and for the true move between the two frames, in the same form as the grid moves (no rotation, gripper unchanged)
action_samples = make_action_grid(GRID_SIZE, GRID_POINTS)
true_move = torch.zeros(1, 1, 7)
true_move[0, 0, :3] = torch.tensor(np_actions[0, 0, :3])
print(f"Sampled grid of actions; num actions = {len(action_samples)}")
with torch.no_grad(), autocast(), measure("energy_grid"):
    h = forward_target(clips)
    loss = energy(h, action_samples)  # jepa prediction loss
with torch.no_grad(), autocast():
    true_move_energy = energy(h, true_move)[0]


def error_cm(move):
    """Distance in cm between the x, y, z of a move and the true move."""
    return float(np.linalg.norm(np.asarray(move[:3], dtype=float) - np_actions[0, 0, :3])) * 100


best = int(np.argmin(loss))
best_move = action_samples[best, 0].numpy()
lower = int(np.sum(np.array(loss) < true_move_energy))  # grid moves with a lower energy than the true move
gt_x, gt_y, gt_z = np_actions[0, 0, :3]
print(f"Ground truth action (x,y,z) = ({gt_x:.3f},{gt_y:.3f},{gt_z:.3f}), energy {true_move_energy:.4f}; "
      f"{lower} of {len(loss)} grid moves have a lower energy")
print(f"Lowest energy on the grid at ({best_move[0]:.3f},{best_move[1]:.3f},{best_move[2]:.3f}), "
      f"energy {loss[best]:.4f}, {error_cm(best_move):.1f} cm from the true move")

In [ ]:
# Plot the energy: for each (Delta x, Delta z), the lowest energy over Delta y

energies = np.array(loss).reshape(GRID_POINTS, GRID_POINTS, GRID_POINTS)  # [x, y, z], as in make_action_grid
step = 2 * GRID_SIZE / max(GRID_POINTS - 1, 1)
edge = GRID_SIZE + step / 2

# Set axis labels
plt.xlabel("Action Delta x (m)")
plt.ylabel("Action Delta z (m)")
plt.title(f"Energy Landscape (lowest over Delta y)")

# Display the heatmap, the true move and the lowest energy on the grid
_ = plt.imshow(energies.min(axis=1).T, origin="lower", extent=[-edge, edge, -edge, edge], cmap="viridis")
_ = plt.colorbar()
_ = plt.scatter([gt_x], [gt_z], marker="x", color="red", label="true move")
_ = plt.scatter([best_move[0]], [best_move[2]], marker="o", facecolors="none", edgecolors="white",
                label="lowest energy on the grid")
_ = plt.legend(loc="upper left", fontsize="small")

In [ ]:
# Compute the optimal action using MPC
from utils.world_model_wrapper import WorldModel

world_model = WorldModel(
    encoder=encoder,
    predictor=predictor,
    tokens_per_frame=tokens_per_frame,
    transform=transform,
    # Doing very few CEM iterations with very few samples just to run efficiently on CPU...
    # ... increase cem_steps and samples for more accurate optimization of energy landscape
    mpc_args={
        "rollout": 2,
        "samples": CEM_SAMPLES,
        "topk": 10,
        "cem_steps": CEM_STEPS,
        "momentum_mean": 0.15,
        "momentum_mean_gripper": 0.15,
        "momentum_std": 0.75,
        "momentum_std_gripper": 0.15,
        "maxnorm": 0.075,
        "verbose": True
    },
    normalize_reps=True,
    device=DEVICE
)

with torch.no_grad(), autocast(), measure("cem"):
    h = forward_target(clips)
    z_n, z_goal = h[:, :tokens_per_frame], h[:, -tokens_per_frame:]
    s_n = states[:, :1]
    print(f"Starting planning using Cross-Entropy Method...")
    actions = world_model.infer_next_action(z_n, s_n, z_goal).float().cpu().numpy()

# CEM plans "rollout" steps, each at most "maxnorm" per axis; the move it plans is the sum of the steps
for i, step_action in enumerate(actions):
    print(f"Step {i + 1} of the plan returned by CEM (x,y,z) = "
          f"({step_action[0]:.3f},{step_action[1]:.3f},{step_action[2]:.3f})")
cem_move = actions[:, :3].sum(axis=0)
print(f"Sum of the {len(actions)} steps (x,y,z) = ({cem_move[0]:.3f},{cem_move[1]:.3f},{cem_move[2]:.3f}), "
      f"{error_cm(cem_move):.1f} cm from the true move")

In [ ]:
# Measurements: time, peak memory, and the moves found compared with the true move
def xyz(move):
    return [round(float(x), 3) for x in move[:3]]


measurements.update({
    "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name() if DEVICE == "cuda" else None,
    "grid_size_m": GRID_SIZE,
    "grid_points": GRID_POINTS,
    "grid_moves": len(action_samples),
    "energy_grid_ms_per_move": round(measurements["energy_grid_s"] / len(action_samples) * 1000, 2),
    "cem_samples": CEM_SAMPLES,
    "cem_steps": CEM_STEPS,
    "cem_rollout": len(actions),
    "seed": SEED,
    "params_millions": {
        "encoder": round(sum(p.numel() for p in encoder.parameters()) / 1e6),
        "predictor": round(sum(p.numel() for p in predictor.parameters()) / 1e6),
    },
    "peak_ram_gib": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20, 2),
    "true_move_xyz": xyz(np_actions[0, 0]),
    "true_move_energy": round(true_move_energy, 4),
    "grid_moves_below_true_move": lower,
    "grid_best_move_xyz": xyz(best_move),
    "grid_best_energy": round(loss[best], 4),
    "grid_best_error_cm": round(error_cm(best_move), 1),
    "cem_plan_xyz": [xyz(a) for a in actions],
    "cem_move_xyz": xyz(cem_move),
    "cem_move_error_cm": round(error_cm(cem_move), 1),
})
print(json.dumps(measurements, indent=2))

if RESULTS_DIR:
    path = os.path.join(RESULTS_DIR, f"energy_landscape_{DEVICE}_{measurements['dtype']}.json")
    with open(path, "w") as f:
        json.dump(measurements, f, indent=2)
    print(f"Saved to {path}")